# UR5e Physical AI：第1回 AI推論演習

目標：日本語の指示を学習済みAIモデルで比較し、`LEFT_SAMPLE`・`RIGHT_SAMPLE`・`HOME` のいずれかを選択する。

今回実装するのは**作業IDの推論のみ**です。ROS 2やUR5eの制御にはまだ接続しません。

モデル：`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`（Hugging Face）。初回はダウンロードが必要です。

以下のセルを **上から順番に ▶ 実行** してください。ランタイムはCPUで構いません。


In [1]:
%pip -q install sentence-transformers


## 1. 学習済みモデルを読み込み、候補タスクを定義
左・右・待機の3種類を、各2つの表現で登録します。


In [2]:
"""Japanese instruction -> task ID by pretrained sentence embeddings.

This is a *task selector*, not a robot controller. Always verify task IDs and
workspace safety before connecting this code to a robot or simulator.
"""
from time import perf_counter

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TASK_EXAMPLES = {
    "LEFT_SAMPLE": [
        "左側に置かれた試料を測定位置へ運んでください",
        "試料Aを検査場所に移してください",
    ],
    "RIGHT_SAMPLE": [
        "右側に置かれた試料を測定位置へ運んでください",
        "試料Bを検査場所に移してください",
    ],
    "HOME": [
        "ロボットアームを待機位置に戻してください",
        "ロボットを初期姿勢に復帰させてください",
    ],
}

model = SentenceTransformer(MODEL_ID)
labels = list(TASK_EXAMPLES)
examples = [(label, text) for label, texts in TASK_EXAMPLES.items() for text in texts]
reference_embeddings = model.encode(
    [text for _, text in examples], normalize_embeddings=True, convert_to_numpy=True
)


def predict(instruction: str):
    start = perf_counter()
    embedding = model.encode([instruction], normalize_embeddings=True, convert_to_numpy=True)[0]
    similarities = reference_embeddings @ embedding
    # For each task, retain its closest reference example.
    scores = {label: max(float(similarities[i]) for i, (name, _) in enumerate(examples) if name == label)
              for label in labels}
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[0], scores, scores[ranked[0]] - scores[ranked[1]], perf_counter() - start




modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 2. 好きな指示文で推論
`instruction` を書き換えて繰り返し実行してください。類似度は確率や安全保証を意味しません。


In [3]:
instruction = "右側の部品を測定位置へ移して"
predicted, scores, margin, seconds = predict(instruction)
print("入力:", instruction)
print("AIが選んだ作業:", predicted)
for task, score in scores.items():
    print(f"  {task}: {score:.4f}")
print(f"1位と2位の差: {margin:.4f}")
print(f"推論時間: {seconds:.3f} 秒")


入力: 右側の部品を測定位置へ移して
AIが選んだ作業: RIGHT_SAMPLE
  LEFT_SAMPLE: 0.8034
  RIGHT_SAMPLE: 0.8157
  HOME: 0.4651
1位と2位の差: 0.0123
推論時間: 0.190 秒


## 3. 6件で評価してCSVを保存
結果は実行時に計算されます。正答率を事前に保証するものではありません。誤分類した場合も重要な実験結果です。


In [4]:
trials = [
    ("左の試料を測定場所へ移動して", "LEFT_SAMPLE"),
    ("右側の部品を検査台まで運んで", "RIGHT_SAMPLE"),
    ("試料Aを検査位置に移して", "LEFT_SAMPLE"),
    ("試料Bを測定位置へ運んで", "RIGHT_SAMPLE"),
    ("アームを待機位置に戻して", "HOME"),
    ("最初の姿勢に戻ってください", "HOME"),
]
rows = []
for instruction, expected in trials:
    chosen, scores, margin, duration = predict(instruction)
    rows.append({
        "instruction": instruction,
        "expected": expected,
        "predicted": chosen,
        "correct": int(chosen == expected),
        "margin": round(margin, 4),
        "inference_seconds": round(duration, 4),
        **{f"score_{key}": round(val, 4) for key, val in scores.items()},
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"正答率：{df['correct'].mean():.1%}（{int(df['correct'].sum())}/{len(df)}件）")
df.to_csv("ai_inference_results.csv", index=False, encoding="utf-8-sig")
print("評価結果を ai_inference_results.csv に保存しました")


   instruction     expected    predicted  correct  margin  inference_seconds  score_LEFT_SAMPLE  score_RIGHT_SAMPLE  score_HOME
左の試料を測定場所へ移動して  LEFT_SAMPLE  LEFT_SAMPLE        1  0.0510             0.0426             0.9520              0.9009      0.4116
右側の部品を検査台まで運んで RIGHT_SAMPLE  LEFT_SAMPLE        0  0.0090             0.0321             0.7216              0.7126      0.5399
  試料Aを検査位置に移して  LEFT_SAMPLE  LEFT_SAMPLE        1  0.0778             0.0309             0.9620              0.8842      0.3655
  試料Bを測定位置へ運んで RIGHT_SAMPLE RIGHT_SAMPLE        1  0.0815             0.0303             0.7755              0.8570      0.3308
  アームを待機位置に戻して         HOME         HOME        1  0.3865             0.0308             0.3556              0.3395      0.7421
 最初の姿勢に戻ってください         HOME         HOME        1  0.2985             0.0289             0.3740              0.3709      0.6726
正答率：83.3%（5/6件）
評価結果を ai_inference_results.csv に保存しました


In [5]:
# ColabのPCにCSVを保存したいときだけ実行
from google.colab import files
files.download("ai_inference_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. 次のステップ
Colabの「ファイル」→「ダウンロード」→「.ipynb をダウンロード」でノートブックを保存し、GitHubリポジトリ `ur5e-physical-ai-2026` にアップロードします。

### 留意事項
- 文章埋め込みの類似度で事前登録タスクを選択しており、ロボットの関節角・経路をAIが直接生成しているわけではありません。
- あいまいな指示は誤認識する場合があります。現段階では選択結果をROS 2へ自動送信しません。
- 入力文、正答ラベル、実行結果を公開する際は個人情報・機密情報がないか確認してください。
